In [1]:
from clustering.clustering import create_cluster
import polars as pl
from genai.mistral import get_mistral_cluster_tags, ClusterWithTags
import time

folder = "ribier_v1"

df = pl.read_parquet(f"./books_work/{folder}/data/{folder}.parquet")
docs = df["embedding"].to_list()
print("Creating clusters...")
cluster, soft, samples, iters = create_cluster(docs, run_num=1)
df = df.with_columns(
    pl.Series("cluster", cluster),
    pl.Series("soft_cluster", soft),
)

print(f"Clustered into {len(set(cluster))} clusters in {iters} iterations.")

print("Getting tags for clusters...")

# Now let's get the cluster tags for the main clusters using Mistral
sample_docs: list[ClusterWithTags] = [
    ClusterWithTags(label=i, tags=df[s].select("markdown").to_series().to_list())
    for i, s in enumerate(samples)
]

res = get_mistral_cluster_tags(sample_docs)
res_dict = {item.label: ", ".join(item.tags) for item in res}
res_list = [res_dict.get(c) for c in cluster]
df = df.with_columns(pl.Series("cluster_tags", res_list))

print(
    "Done getting cluster tags. Now calculating sub-clusters for clusters with enough samples..."
)


print("Calculating cluster counts and percentages...")
n = len(df)
cluster_counts = df["cluster"].value_counts().sort("cluster")
cluster_counts = cluster_counts.with_columns(cluster_perc=cluster_counts["count"] / n)
clusters_to_sub_df = cluster_counts.filter(pl.col("count") ** (1 / 3) >= 3)
clusters_to_sub = clusters_to_sub_df["cluster"].to_list()
df = df.with_columns(sub_cluster=pl.lit(-1).cast(pl.Int64))
df = df.with_columns(sub_labels=pl.lit(None).cast(pl.List(pl.Utf8)))

print(f"Clusters to sub-cluster: {clusters_to_sub}")
# Let's get sub-clusters for clusters that have enough samples, and get their tags using Mistral
for c in clusters_to_sub:
    cluster_count = clusters_to_sub_df.filter(pl.col("cluster") == c)["count"][0] ** (
        1 / 3
    )
    parent_cluster_labels = res_dict.get(c)
    cluster_count = int(cluster_count)
    sub_df = df.filter(pl.col("cluster") == c)
    sub_docs = sub_df["embedding"].to_list()
    sub_cluster, _, sub_samples, iter = create_cluster(
        sub_docs, run_num=1, soft_assign=False, num_clusters=cluster_count
    )
    print(f"Sub-cluster {c} created with {iter} iterations.")
    # Get sub-cluster tags using Mistral
    print(f"Getting tags for sub-cluster {c}...")
    sample_docs: list[ClusterWithTags] = [
        ClusterWithTags(label=i, tags=df[s].select("markdown").to_series().to_list())
        for i, s in enumerate(sub_samples)
    ]
    res = get_mistral_cluster_tags(
        sample_docs, is_sub_cluster=True, parent_cluster_labels=parent_cluster_labels
    )
    sub_samples_dict = {item.label: ", ".join(item.tags) for item in res}
    sub_samples_list = [sub_samples_dict.get(c) for c in sub_cluster]

    # Assign sub_cluster to sub_df
    sub_df = sub_df.with_columns(pl.Series("sub_c", sub_cluster))
    sub_df = sub_df.with_columns(pl.Series("sub_s", sub_samples_list))

    df = df.join(
        sub_df.select(["chunk_id", "sub_c", "sub_s"]), on="chunk_id", how="left"
    )
    df = df.with_columns(
        sub_cluster=(
            pl.when(pl.col("sub_c").is_not_null())
            .then(pl.col("sub_c"))
            .otherwise(pl.col("sub_cluster"))
        ),
        sub_labels=(
            pl.when(pl.col("sub_s").is_not_null())
            .then(pl.col("sub_s"))
            .otherwise(pl.col("sub_labels"))
        ),
    )
    df = df.drop("sub_c")
    df = df.drop("sub_s")
    # If you want to fill nulls with a default value (e.g., -1), uncomment:
    df = df.with_columns(pl.col("sub_cluster").fill_null(-1))


df.write_parquet(f"./books_work/{folder}/data/{folder}.parquet")
df

Creating clusters...
  Estimated number of clusters: 14
  Restart 1/10 (seed 10) — score: 0.9375
  Restart 2/10 (seed 11) — score: 0.9371
  Restart 3/10 (seed 12) — score: 0.9384
  Restart 4/10 (seed 13) — score: 0.9376
  Restart 5/10 (seed 14) — score: 0.9375
  Restart 6/10 (seed 15) — score: 0.9376
  Restart 7/10 (seed 16) — score: 0.9374
  Restart 8/10 (seed 17) — score: 0.9372
  Restart 9/10 (seed 18) — score: 0.9374
  Restart 10/10 (seed 19) — score: 0.9379
  Best score: 0.9384
Clustered into 14 clusters in 15 iterations.
Getting tags for clusters...
Using Mistral model: mistral-large-latest. Instituting a delay of 4 seconds between API calls to avoid rate limits.
Done getting cluster tags. Now calculating sub-clusters for clusters with enough samples...
Calculating cluster counts and percentages...
Clusters to sub-cluster: [0, 2, 12]
  Restart 1/10 (seed 10) — score: 0.9430
  Restart 2/10 (seed 11) — score: 0.9426
  Restart 3/10 (seed 12) — score: 0.9433
  Restart 4/10 (seed 13) 

letter_id,id,markdown,pdf_pages,title,chunk_id,word_length,full_markdown,embedding,cluster,soft_cluster,sub_cluster,sub_labels,cluster_tags
u32,u32,str,list[i64],str,f64,i64,str,list[f64],i64,list[i64],i64,str,str
10000,1000,"""LE PAPE PAVL III. AV ROT FRANC…",[46],"""LE PAPE PAVL III. AV ROT FRANC…",10000.0,176,"""LE PAPE PAVL III. AV ROT FRANC…","[0.024414, 0.036377, … -0.024536]",10,[10],-1,null,"""PapalDiplomacy, Ecclesiastical…"
10001,1001,"""LETTRE DE MONLVC AV CARDINAL D…","[46, 47]","""LETTRE DE MONLVC AV CARDINAL D…",10001.0,470,"""LETTRE DE MONLVC AV CARDINAL D…","[-0.00296, 0.011841, … -0.024292]",2,"[2, 9, 12]",2,"""PapalMediationTactics, Strateg…","""PapalDiplomacy, HighPoliticalC…"
10002,1002,"""LETTRE DES AVOTERS ET CONSEILL…","[47, 48]","""LETTRE DES AVOTERS ET CONSEILL…",10002.0,452,"""LETTRE DES AVOTERS ET CONSEILL…","[-0.0354, -0.02771, … 0.023071]",12,"[2, 9, 12]",2,"""DiplomaticTerritorialDisputes,…","""DiplomaticCorrespondence, Roya…"
10003,1003,"""REMARQUES SVR LADITE VILLE DE …","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.0,1878,"""REMARQUES SVR LADITE VILLE DE …","[-0.036621, -0.010559, … -0.013855]",8,[8],-1,null,"""DiplomaticAllianceNegotiations…"
10003,1003,"""Cethol. 1192. 6 commiæer Janoe…","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.1,778,"""REMARQUES SVR LADITE VILLE DE …","[0.007721, -0.031738, … -0.024902]",10,"[10, 13]",-1,null,"""PapalDiplomacy, Ecclesiastical…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
10261,1261,"""DE L'AUBESPINE. --- ## LE ROY…","[656, 657]","""DE L'AUBESPINE.""",10261.0,432,"""DE L'AUBESPINE. --- ## LE ROY…","[-0.04541, -0.05542, … 0.011902]",1,[1],-1,null,"""ImperialProtestantConflict, Di…"
10262,1262,"""# A MONSIEUR LE DAVHIN. Le Ca…","[657, 658]","""A MONSIEUR LE DAVHIN.""",10262.0,498,"""# A MONSIEUR LE DAVHIN. Le Ca…","[-0.004944, 0.012268, … -0.06543]",2,"[0, 2, 3]",2,"""PapalMediationTactics, Strateg…","""PapalDiplomacy, HighPoliticalC…"
10263,1263,"""LE CARDINAL DE BOLOGNE ## AV R…","[658, 659]","""LE CARDINAL DE BOLOGNE AV ROT…",10263.0,560,"""LE CARDINAL DE BOLOGNE ## AV R…","[-0.036133, -0.028687, … -0.050781]",0,[0],0,"""PapalDiplomaticEndorsement, It…","""DiplomaticNegotiations, PapalI…"


In [2]:
clusters_w_subs = [0, 2, 12]

for i in clusters_w_subs:
    print("---" * 5)
    print(f"Cluster {i} tags:")
    clust_df = df.filter(pl.col("cluster") == i)
    print(clust_df["cluster_tags"].unique().to_list())
    print(clust_df["sub_cluster"].value_counts())
    for sub_c in clust_df["sub_cluster"].unique():
        print(f"\tSub-cluster {sub_c} tags:")
        print(
            clust_df.filter(pl.col("sub_cluster") == sub_c)["sub_labels"]
            .unique()
            .to_list()
        )

---------------
Cluster 0 tags:
['DiplomaticNegotiations, PapalImperialAlliances, AntiTurkishLeague, VenetianGeopolitics, DynasticTreatyFormation']
shape: (3, 2)
┌─────────────┬───────┐
│ sub_cluster ┆ count │
│ ---         ┆ ---   │
│ i64         ┆ u32   │
╞═════════════╪═══════╡
│ 0           ┆ 27    │
│ 2           ┆ 2     │
│ 1           ┆ 11    │
└─────────────┴───────┘
	Sub-cluster 0 tags:
['PapalDiplomaticEndorsement, ItalianCityStateAlliances, AntiImperialFlorentineRestoration, MercenaryTacticalOperations, FrancoPapalMilitaryCoordination']
	Sub-cluster 1 tags:
['PapalPeaceMediation, AntiImperialDiplomaticTactics, SecretNegotiationChannels, EcclesiasticalPoliticalLeverage, FrancoPapalDistrustDynamic']
	Sub-cluster 2 tags:
['EcclesiasticalAutonomyDebate, EpiscopalResidencyAdvocacy, AntiPatronageMoralism, HeresyModerationPolicy, ChurchReformFromBelow']
---------------
Cluster 2 tags:
['PapalDiplomacy, HighPoliticalCorrespondence, CardinalMinisters, FrancoImperialRivalry, Ecclesias

In [4]:
cluster = 12
subcluster = 0

cluster12_samples = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["markdown"])
    .sample(8)
    .to_series()
    .to_list()
)
parent_labels = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["cluster_tags"])
    .unique()
    .to_series()
    .to_list()
)
sub_labels = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["sub_labels"])
    .unique()
    .to_series()
    .to_list()
)
print(f" Cluster {cluster}, Sub-cluster {subcluster} samples:")
print(cluster12_samples)
print(" Parent labels:")
print(parent_labels)
print(" Sub-labels:")
print(sub_labels)

 Cluster 12, Sub-cluster 0 samples:
["Z ij\n\n20\n\nque les autres Roys messnes luy dénuient; fay une procuration Lazine du Roy François premier, datée du 25. Mar. 1327. addossante au Seigneur de Montmorency et autres, pour aller en Angleterre jurer la Paix entre les deux Couronnes, au lotir Henry est qualifé DEVENSOR FIDEZ. Il avoit encore beaucoup merué du S. Siège par sa franche et volontaire jonction et contribution à l'armée de France, pour la délivrance du Pape Clement septième, que l'Empereur renoit prisonnier dans le Château S. Ange, enfin son manuais sent et ses furieux emportemens n'allerent pas jusqu'à l'Heresse, mais bien au Sépime formé, qui n'est pas moins dangereux et pernicieux que l'autre, comme remarquent fort judicieusement divers Peres anciens de l'Eglise, et même qu'il est ordinairement juius de l'Heresse : un en voit en celuy-cy l'exemple; car apres la mort d'Henry huitième avriuse au commencement de 1347. elle s'établit puiffamment sous Edouard son Fils et succét

In [5]:
import polars as pl

df = pl.read_parquet(f"./books_work/ribier_v1/data/ribier_v1.parquet")
df.head()

letter_id,id,markdown,pdf_pages,title,chunk_id,word_length,full_markdown,embedding,cluster,soft_cluster,sub_cluster,sub_labels,cluster_tags
u32,u32,str,list[i64],str,f64,i64,str,list[f64],i64,list[i64],i64,list[str],list[str]
10000,1000,"""LE PAPE PAVL III. AV ROT FRANC…",[46],"""LE PAPE PAVL III. AV ROT FRANC…",10000.0,176,"""LE PAPE PAVL III. AV ROT FRANC…","[0.024414, 0.036377, … -0.024536]",10,[10],-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"
10001,1001,"""LETTRE DE MONLVC AV CARDINAL D…","[46, 47]","""LETTRE DE MONLVC AV CARDINAL D…",10001.0,470,"""LETTRE DE MONLVC AV CARDINAL D…","[-0.00296, 0.011841, … -0.024292]",2,"[2, 9, 12]",3,"[""PapalMediationAttempts"", ""SuspicionOfDividedLoyalties"", … ""TruceNegotiations""]","[""PapalDiplomacy"", ""NeutralityStrategies"", … ""FrenchMediation""]"
10002,1002,"""LETTRE DES AVOTERS ET CONSEILL…","[47, 48]","""LETTRE DES AVOTERS ET CONSEILL…",10002.0,452,"""LETTRE DES AVOTERS ET CONSEILL…","[-0.0354, -0.02771, … 0.023071]",12,"[2, 9, 12]",0,"[""DiplomaticNegotiations"", ""SuzerainVassalTensions"", … ""StateSovereigntyClaims""]","[""DiplomaticEspionage"", ""RenaissanceStatecraft"", … ""CourtlyIntrigue""]"
10003,1003,"""REMARQUES SVR LADITE VILLE DE …","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.0,1878,"""REMARQUES SVR LADITE VILLE DE …","[-0.036621, -0.010559, … -0.013855]",8,[8],-1,null,"[""DiplomaticNegotiations"", ""ConfessionalFragmentation"", … ""ReligiousDissent""]"
10003,1003,"""Cethol. 1192. 6 commiæer Janoe…","[48, 49, … 54]","""REMARQUES SVR LADITE VILLE DE …",10003.1,778,"""REMARQUES SVR LADITE VILLE DE …","[0.007721, -0.031738, … -0.024902]",10,"[10, 13]",-1,null,"[""PapalConciliarDiplomacy"", ""RhetoricOfPapalAuthority"", … ""PrincelyMediationEfforts""]"


In [1]:
import polars as pl

# df = pl.read_parquet(f"./books_work/ribier_v1/data/ribier_v1.parquet")

In [4]:
clusters_w_subs = [0, 2, 12]
from clustering.clustering import (
    analyze_geometric_footprint,
    recommend_subclustering,
    recommend_k,
    recommend_k_refined,
)

for i in clusters_w_subs:
    print("---" * 5)
    print(f"Cluster {i} recommendation:")
    clust_df = df.filter(pl.col("cluster") == i)
    vectors = clust_df["embedding"]
    analyze_geometric_footprint(vectors, cluster_label=f"Cluster {i}")
    recommend_k(vectors, cluster_label=f"Cluster {i}")
    k = recommend_k_refined(vectors, cluster_label=f"Cluster {i}")
    print(f"Recommended k for Cluster {i}: {k}")

---------------
Cluster 0 recommendation:

Geometric Analysis: Cluster 0
  Total Samples: 40
  Dimensions to explain 80% variance: 21
  Anisotropy (Axis 1 / Axis 2): 1.02
  >>> Recommendation: SUB-CLUSTER (Occupies broad conceptual territory)

Sub-clustering Logic for Cluster 0:
  Total Dimensions available: 40
  Significant Axes (> average): 15
  Axes to reach 50% variance: 10
  >>> RECOMMENDED k: 6
Recommended k for Cluster 0: 2
---------------
Cluster 2 recommendation:

Geometric Analysis: Cluster 2
  Total Samples: 97
  Dimensions to explain 80% variance: 37
  Anisotropy (Axis 1 / Axis 2): 1.08
  >>> Recommendation: SUB-CLUSTER (Occupies broad conceptual territory)

Sub-clustering Logic for Cluster 2:
  Total Dimensions available: 97
  Significant Axes (> average): 31
  Axes to reach 50% variance: 15
  >>> RECOMMENDED k: 6
Recommended k for Cluster 2: 2
---------------
Cluster 12 recommendation:

Geometric Analysis: Cluster 12
  Total Samples: 61
  Dimensions to explain 80% varianc

In [3]:
all_vectors = df["embedding"]
analyze_geometric_footprint(all_vectors, cluster_label="All data")
recommend_k(all_vectors, cluster_label="All data")
recommend_k_refined(all_vectors, cluster_label="All data")


Geometric Analysis: All data
  Total Samples: 338
  Dimensions to explain 80% variance: 61
  Anisotropy (Axis 1 / Axis 2): 1.05
  >>> Recommendation: SUB-CLUSTER (Occupies broad conceptual territory)

Sub-clustering Logic for All data:
  Total Dimensions available: 338
  Significant Axes (> average): 81
  Axes to reach 50% variance: 22
  >>> RECOMMENDED k: 6


2

In [2]:
clusters_w_subs = [0, 2, 12]

for i in clusters_w_subs:
    print("---" * 5)
    print(f"Cluster {i} tags:")
    clust_df = df.filter(pl.col("cluster") == i)
    print(clust_df["cluster_tags"].unique().to_list())
    print(clust_df["sub_cluster"].value_counts())
    for sub_c in clust_df["sub_cluster"].unique():
        print(f"\tSub-cluster {sub_c} tags:")
        print(
            clust_df.filter(pl.col("sub_cluster") == sub_c)["sub_labels"]
            .unique()
            .to_list()
        )

---------------
Cluster 0 tags:
['DiplomaticAlliancesAndTreaties, PapalImperialVenetianRelations, OttomanConflictAndMediation, DynasticSuccessionAndProtection, MilitaryStrategyAndEspionage']
shape: (3, 2)
┌─────────────┬───────┐
│ sub_cluster ┆ count │
│ ---         ┆ ---   │
│ i64         ┆ u32   │
╞═════════════╪═══════╡
│ 0           ┆ 27    │
│ 1           ┆ 11    │
│ 2           ┆ 2     │
└─────────────┴───────┘
	Sub-cluster 0 tags:
['FrenchPapalDiplomaticIntermediaries, FlorenceTuscanyFrontier, CovertRegimeChange, MediciSuccessionVulnerability, CardinalAgentNetworks']
	Sub-cluster 1 tags:
['PapalLegationManipulation, ItalianPeninsulaFrontier, SubversiveMediation, PapalNeutralityErosion, LegateDoubleAgency']
	Sub-cluster 2 tags:
['CardinalCarpentrasResistance, RomanCurialPensions, EpiscopalAutonomyAssertion, TheologicalNonViolentReformation, EcclesiasticalPatronageRejection']
---------------
Cluster 2 tags:
['PapalDiplomaticNegotiations, ItalianGeopoliticalAlliances, CardinalMinis

In [14]:
cluster = 2
subcluster = 3

cluster12_samples = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["markdown"])
    .sample(5)
    .to_series()
    .to_list()
)
parent_labels = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["cluster_tags"])
    .unique()
    .to_series()
    .to_list()
)
sub_labels = (
    df.filter((pl.col("cluster") == cluster) & (pl.col("sub_cluster") == subcluster))
    .select(["sub_labels"])
    .unique()
    .to_series()
    .to_list()
)
print(f" Cluster {cluster}, Sub-cluster {subcluster} samples:")
print(cluster12_samples)
print(" Parent labels:")
print(parent_labels)
print(" Sub-labels:")
print(sub_labels)

 Cluster 2, Sub-cluster 3 samples:
["# TRAITTE DE TR'EVE\n\nfait à Bomy. A L'HONNEUR de Dieu noître Cteateur : Comme pour éviter lexion du sang humain &amp; faire ceffer les grands maus &amp; innumérables inconveniens qui prouiennent à l'occasion de la guerre, &amp; pour mieux parvenir à vue bonne paix finale se foient assemblés, entri en communication Meffite Ican Dalbon, Seigneur de saint André Confeillet Chambellan ordinaire du Roy Tres-Chreftien &amp; Chevalier de son Ordre, Maiffte Guillaume Poyet, auffi Chevalier Confeillet dudit Seigneur en son priué &amp; secret Confeil, Prefident en sa Cour de Parlement à Paris, premier Prefident de Bretagne, Seigneur du Coudray, &amp; Maiffte Nicolas Betheteau : Notaire &amp; Secrétaire d'Estat &amp; de Chambre dudit Seigneur, Commis &amp; Deputés par tteshaut &amp; puissant Prince Monseigneur Henry premier fils dudit Seigneur Roy, D. aphin de Viennois, Duc de Bretagne, Comte de Valentinois &amp; de Diois, Gouverneut de Normandie &amp; Lieute

In [13]:
sadolet = df.filter(pl.col("markdown").str.contains("sadol", literal=True))
sadolet

letter_id,id,markdown,pdf_pages,title,chunk_id,word_length,full_markdown,embedding,cluster,soft_cluster,sub_cluster,sub_labels,cluster_tags
u32,u32,str,list[i64],str,f64,i64,str,list[f64],i64,list[i64],i64,str,str


In [ ]:
cluster9_sampels = (
    df.filter(pl.col("cluster1") == 9)
    .select("markdown")
    .sample(5)
    .to_series()
    .to_list()
)
cluster9_sampels

ColumnNotFoundError: unable to find column "cluster1"; valid columns: ["letter_id", "id", "markdown", "pdf_pages", "title", "chunk_id", "word_length", "full_markdown", "embedding", "cluster", "soft_cluster", "sub_cluster", "sub_labels", "cluster_tags"]

In [9]:
cluster12_samples = (
    df.filter(pl.col("cluster1") == 12)
    .select("markdown")
    .sample(5)
    .to_series()
    .to_list()
)
cluster12_samples

["\n\nD I L E C T E fili, salutem &amp; Apostolica m benedictionem. Audito Christianiflimi Regis sapienti judicio, quo nobilitatem tuam ad supremum Regni Francie Magistratum evexit, magna profecto latitia affecti sumus. Quot viro &amp; probo, &amp; prudenti, &amp; Q. iij\n\nOriginal du 11. Mars.3\n\nLe ferey voir cyapres, par divers les Lettres, que les Papes traitent les Ducs même fou. Certains de Nobiliu vis &amp; de Nobilius.\n\nLes Eloges du Pape &amp; du Connoissable se veront cyapres.\n\nOriginal.\n\n\n\n# LETTRES ET MEMOIRES DESTAT.\n\nmultis virtutibus ornato, ex potestas accellerit, quâ &amp; rei Christianae, &amp; Apostolicae Sedis possit maiori multo auctoritate patrocinari. Qualem mentem, animum qu in omnibus actionibus suis temper ostendit. Graculamur itaq; primum Regia Majestati, quæ beneficiu dando accepit, cum digno dedit. Deinde Nobilitati tuæ, quæ tantum Principis iudicium in se promeruit. Er speramus hoc, Deo benevolente atq; auctore, factum esse, vit ea Fidelium rem

In [7]:
from clustering.clustering import create_cluster
import polars as pl

folder = "ribier_v1"

df1 = pl.read_parquet(f"./books_work/{folder}/data/{folder}_chunked.parquet")
docs = df1["embedding"].to_list()
cluster3, soft3, samples3, iters3 = create_cluster(docs, run_num=1, soft_assign=False)
cluster4, soft4, samples4, iters4 = create_cluster(docs, run_num=2, soft_assign=False)
df1 = df1.with_columns(cluster3=pl.Series(cluster3), cluster4=pl.Series(cluster4))
print("Cluster 3:")
with pl.Config(tbl_rows=25):
    display(df1["cluster3"].value_counts().sort("cluster3"))
print("Cluster 4:")
with pl.Config(tbl_rows=25):
    display(df1["cluster4"].value_counts().sort("cluster4"))

  Estimated number of clusters: 14
  Restart 1/10 (seed 10) — score: 0.9402
  Restart 2/10 (seed 11) — score: 0.9400
  Restart 3/10 (seed 12) — score: 0.9403
  Restart 4/10 (seed 13) — score: 0.9403
  Restart 5/10 (seed 14) — score: 0.9398
  Restart 6/10 (seed 15) — score: 0.9402
  Restart 7/10 (seed 16) — score: 0.9404
  Restart 8/10 (seed 17) — score: 0.9399
  Restart 9/10 (seed 18) — score: 0.9401
  Restart 10/10 (seed 19) — score: 0.9393
  Best score: 0.9404
  Estimated number of clusters: 14
  Restart 1/10 (seed 20) — score: 0.9406
  Restart 2/10 (seed 21) — score: 0.9400
  Restart 3/10 (seed 22) — score: 0.9396
  Restart 4/10 (seed 23) — score: 0.9399
  Restart 5/10 (seed 24) — score: 0.9401
  Restart 6/10 (seed 25) — score: 0.9394
  Restart 7/10 (seed 26) — score: 0.9403
  Restart 8/10 (seed 27) — score: 0.9402
  Restart 9/10 (seed 28) — score: 0.9386
  Restart 10/10 (seed 29) — score: 0.9399
  Best score: 0.9406
Cluster 3:


cluster3,count
i64,u32
0,75
1,21
2,17
3,78
4,6
5,26
6,23
7,9
8,3


Cluster 4:


cluster4,count
i64,u32
0,28
1,65
2,19
3,9
4,23
5,6
6,37
7,68
8,11


In [11]:
df.sample(5)

letter_id,id,title,pdf_pages,book_pages,markdown,chunk_id,word_length,full_markdown,embedding,cluster1,cluster2
u32,u32,str,list[i64],list[str],str,f64,i64,str,list[f64],i64,i64
10129,10129,"""APOLOGIE""","[327, 328, … 356]","[""104"", ""105"", … ""333""]","""Nous en avons des témoignages …",10129.6,1863,""" des permissions de Duel obte…","[-0.043457, -0.029053, … -0.014771]",8,11
10082,10082,"""LA CHAMBRE DES COMPTES.""","[226, 227]","[""103"", ""204""]",""" Elle depute tant vers le Roy…",10082.0,146,""" Elle depute tant vers le Roy…","[-0.014221, -0.008301, … -0.027832]",12,8
10050,10050,"""AVDIT CONNESTABLE""",[174],[],""" J. Jacques Castron Commis à …",10050.0,64,""" J. Jacques Castron Commis à …","[-0.025146, -0.019775, … 0.001678]",4,0
10022,10022,"""AVTRES ARTICLES DÉBATTVS""","[82, 83, … 85]","[""55"", ""63"", ""2""]",""" en ladite communication non …",10022.0,1570,""" en ladite communication non …","[-0.032715, 0.003464, … 0.00589]",4,11
10271,10271,"""CAROLVS.""",[568],"[""547""]",""" """,10271.0,0,""" ""","[-0.048584, -0.026123, … -0.038818]",1,2


letter_id,id,title,pdf_pages,book_pages,markdown,chunk_id,word_length,full_markdown,embedding,cluster1,cluster2,subcluster1
u32,u32,str,list[i64],list[str],str,f64,i64,str,list[f64],i64,i64,null
10262,10262,"""AV CARDINAL DV BELLAY.""",[554],"[""533""]",""" Sur quelques intrigues de Co…",10262.0,345,""" Sur quelques intrigues de Co…","[-0.047852, -0.026733, … 0.021362]",12,8,null
10292,10292,"""GOSTAVVS.""","[592, 593, … 595]","[""571"", ""52"", … ""374""]",""" Cccc iij52 LETTRES ET MEMOIR…",10292.0,1490,""" Cccc iij52 LETTRES ET MEMOIR…","[-0.044434, -0.009094, … -0.026611]",13,8,null
10009,10009,"""PAVLVS PAPA III.""",[46],"[""15""]",""" CHARISSIME in Christo Fili n…",10009.0,142,""" CHARISSIME in Christo Fili n…","[0.00531, 0.01001, … -0.019653]",11,6,null
10029,10029,"""REMARQUES SVR LEDIT SIEVR""","[116, 117, … 120]","[""23"", ""34"", … ""99""]","""de Selve Ambaffadeur. I'A cy-…",10029.0,1992,"""de Selve Ambaffadeur. I'A cy-…","[-0.001022, -0.030273, … 0.020264]",9,7,null
10006,10006,"""ENSVIT LEDIT PLAIDOYE.""","[27, 28, … 39]","[""6"", ""7"", … ""13""]","""Sire, Varus Orateur insigne vo…",10006.0,1761,""" Sire, Varus Orateur insigne …","[-0.039062, -0.052002, … -0.034424]",12,11,null
